# `wsi2sc` core functions

This notebook is a practical map of the third-party `wsi2sc` submodule. It documents the public functions that this project is most likely to call and provides guarded examples for the local DICOM slide.

The notebook intentionally does not execute the full slide pipeline by default. Whole-slide inference is expensive and writes output files.

## Pipeline boundary

`wsi2sc` currently performs:

1. WSI reading (`DICOM`, OpenSlide formats, or ordinary images).
2. Physical-resolution selection and tiled reading.
3. Optional brightness-based monolayer filtering.
4. Cell-centre detection with NCA or YOLO.
5. Level-0 coordinate mapping and optional crop quality filtering.
6. Optional DinoBloom feature extraction.
7. CSV, NumPy, and profiling output.

It does **not** currently predict biological cell types. The retained cell IDs and WSI coordinates provide the hand-off point for a future cell-type classifier.

### Inference only: no parameter training

The submodule does **not** train or fine-tune model parameters. It performs inference with pretrained parameters: the default NCA detector loads the bundled `models/detection/nca_weights_full` checkpoint, and crop-quality filtering loads the bundled MobileNet checkpoint. Configuration values such as thresholds, update steps, device, and tile size control inference; they do not train the models. YOLO detection and DinoBloom feature extraction require separately supplied pretrained checkpoints when enabled.

In [1]:
import inspect
from pathlib import Path

import wsi2sc
from wsi2sc.cell_processing.factory import build_cell_processors
from wsi2sc.config import config_from_dict, load_config
from wsi2sc.filters.factory import build_filter
from wsi2sc.geometry import LevelInfo, resolve_sampling, scale_coords_to_level0
from wsi2sc.io.factory import open_reader
from wsi2sc.models.factory import build_detector
from wsi2sc.pipeline import run_dataset, run_pipeline


def find_repo_root(start: Path) -> Path:
    """Find the main repository whether Jupyter starts here or in notebooks/."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "third_party" / "wsi2sc"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the leukemia_diagnostics repository root."
    )


REPO_ROOT = find_repo_root(Path.cwd())
WSI2SC_ROOT = REPO_ROOT / "third_party" / "wsi2sc"
PATIENT_ID = "A6BBC91AE73DD21C0533F735470A9CD0"
PATIENT_DIR = REPO_ROOT / "_data" / "bonemarrowwsi_pediatricleukemia" / PATIENT_ID
DICOM_IMAGE_DIR = PATIENT_DIR / "MS"

print(f"wsi2sc {wsi2sc.__version__}")
print(f"package: {Path(wsi2sc.__file__).resolve()}")
print(f"patient image directory exists: {DICOM_IMAGE_DIR.is_dir()}")

wsi2sc 0.1.0
package: C:\Users\Tim Knupfer\Documents\Tum\Single Cell Biology\leukemia_diagnostics\third_party\wsi2sc\src\wsi2sc\__init__.py
patient image directory exists: True


## 1. Configuration

The two main entry points are:

- `load_config(path)`: load and validate YAML.
- `config_from_dict(mapping)`: validate an in-memory mapping.

Unknown keys and invalid types raise `ConfigError`, which helps catch misspelled options. Unspecified values come from the configuration dataclasses.

In [2]:
example_config_path = WSI2SC_ROOT / "configs" / "example.yaml"
example_config = load_config(example_config_path)

print(inspect.signature(load_config))
print(inspect.signature(config_from_dict))
print(example_config)

(path: 'str | Path') -> 'Config'
(raw: 'dict') -> 'Config'
Config(data=DataConfig(input_path='data/example_slides/MRXS/B-other_26_pb.mrxs', input_format='mrxs', base_mpp=None, region=RegionConfig(x=125696, y=136960, width=4096, height=4096, level=0), mode='single', glob=None, inputs=None, list_path=None, recursive=True, extensions=None, dicom_group='leaf'), monolayer_filter=TileFilterConfig(type='brightness', enabled=True, params={'white_thresh': 245, 'black_thresh': 10, 'coverage_thresh': 0.98, 'mixed_thresh': 0.9, 'dark_thresh': 120, 'density_thresh': 0.15}), detection=DetectionConfig(model='nca', weights=None, tile_size=512, overlap=0, batch_size=16, device='auto', level=None, target_mpp=0.97, dedup_radius=None, num_workers=8, params={'radius': 8, 'postprocess_mode': 'max', 'steps': 6, 'fire_rate': 0.5, 'seed': 0}), output=OutputConfig(csv_path=None, dir='data/output', per_input_dir=False, skip_existing=False), profiling=ProfilingConfig(enabled=True, json_path=None), cell_processing

### Project-specific DICOM configuration

Use the patient's `MS/` directory as input so the reader sees only slide-microscopy image objects, not the separate `ANN/` annotation objects. The configuration below is constructed but not executed.

In [3]:
config = config_from_dict(
    {
        "data": {
            "input_path": str(DICOM_IMAGE_DIR),
            "input_format": "dicom",
            "mode": "single",
        },
        "monolayer_filter": {"type": "brightness", "enabled": True},
        "detection": {
            "model": "nca",
            "target_mpp": 0.97,
            "device": "auto",
        },
        "cell_processing": {
            "patch_size": 192,
            "quality": {"enabled": True, "threshold": 0.5},
            "features": {"enabled": False},
        },
        "output": {
            "dir": str(REPO_ROOT / "outputs" / "wsi2sc"),
            "skip_existing": True,
        },
    }
)
config

Config(data=DataConfig(input_path='C:\\Users\\Tim Knupfer\\Documents\\Tum\\Single Cell Biology\\leukemia_diagnostics\\_data\\bonemarrowwsi_pediatricleukemia\\A6BBC91AE73DD21C0533F735470A9CD0\\MS', input_format='dicom', base_mpp=None, region=None, mode='single', glob=None, inputs=None, list_path=None, recursive=True, extensions=None, dicom_group='leaf'), monolayer_filter=TileFilterConfig(type='brightness', enabled=True, params={}), detection=DetectionConfig(model='nca', weights=None, tile_size=512, overlap=0, batch_size=16, device='auto', level=None, target_mpp=0.97, dedup_radius=None, num_workers=8, params={}), output=OutputConfig(csv_path=None, dir='C:\\Users\\Tim Knupfer\\Documents\\Tum\\Single Cell Biology\\leukemia_diagnostics\\outputs\\wsi2sc', per_input_dir=False, skip_existing=True), profiling=ProfilingConfig(enabled=True, json_path=None), cell_processing=CellProcessingConfig(patch_size=192, batch_size=128, device='auto', num_workers=8, quality=QualityProcessingConfig(enabled=Tr

## 2. Slide readers: `open_reader`

`open_reader(path, input_format)` selects a `DicomWSIReader`, `OpenSlideReader`, or `ImageReader`. Every reader exposes the same small interface:

- pyramid level count, dimensions, and downsample
- base microns per pixel (MPP), when available
- `read_region(location, level, size)`
- `close()`

Always close readers. Opening the full slide is guarded below.

In [4]:
OPEN_SLIDE = False

if OPEN_SLIDE:
    reader = open_reader(str(DICOM_IMAGE_DIR), "dicom")
    try:
        levels = [
            LevelInfo(
                level=i,
                downsample=reader.get_level_downsample(i),
                width=reader.get_level_dimensions(i)[0],
                height=reader.get_level_dimensions(i)[1],
            )
            for i in range(reader.get_level_count())
        ]
        print("base MPP:", reader.get_base_mpp())
        for level in levels:
            print(level)
    finally:
        reader.close()
else:
    print("Set OPEN_SLIDE = True to inspect the local DICOM pyramid.")

Set OPEN_SLIDE = True to inspect the local DICOM pyramid.


## 3. Physical sampling and coordinates

`resolve_sampling` chooses a native pyramid level and any residual resize required to reach the detector's target MPP. `scale_coords_to_level0` maps detector-grid coordinates back to the slide's level-0 coordinate system. Correct MPP metadata is essential because it determines the physical scale of detection and cropping.

In [ ]:
synthetic_levels = [
    LevelInfo(level=0, downsample=1.0, width=100_000, height=80_000),
    LevelInfo(level=1, downsample=2.0, width=50_000, height=40_000),
    LevelInfo(level=2, downsample=4.0, width=25_000, height=20_000),
    LevelInfo(level=3, downsample=8.0, width=12_500, height=10_000),
]

sampling = resolve_sampling(
    base_mpp=0.1213,
    levels=synthetic_levels,
    target_mpp=0.97,
)
print(sampling)
print(scale_coords_to_level0([(10, 20)], downsample=8.0))

SamplingPlan(level=3, read_scale=1.0, effective_mpp=0.9704)
[(84, 164)]


: 

## 4. Component factories

The pipeline builds its components from validated configuration:

- `build_filter`: brightness filter or no-op filter.
- `build_detector`: bundled NCA detector or optional YOLO detector.
- `build_cell_processors`: crop quality model and optional DinoBloom feature extractor.

Constructing detectors/processors loads model weights and may allocate GPU memory, so the cell below only displays their signatures.

In [ ]:
core_factories = {
    "build_filter": build_filter,
    "build_detector": build_detector,
    "build_cell_processors": build_cell_processors,
}

for name, function in core_factories.items():
    print(f"{name}{inspect.signature(function)}")

## 5. Main orchestration: `run_pipeline` and `run_dataset`

`run_pipeline(config)` processes one input and returns the path to its cell CSV. `run_dataset(config)` expands folder, glob, or list inputs, continues after individual failures, and returns all successful output paths.

The full run is opt-in below. Start with a small `data.region` in the configuration when validating a new slide or environment.

In [ ]:
print(f"run_pipeline{inspect.signature(run_pipeline)}")
print(f"run_dataset{inspect.signature(run_dataset)}")

RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    if not DICOM_IMAGE_DIR.is_dir():
        raise FileNotFoundError(DICOM_IMAGE_DIR)
    cells_csv = run_pipeline(config)
    print(cells_csv)
else:
    print("Set RUN_FULL_PIPELINE = True only when you intend to run inference.")

## 6. Outputs

A standard run writes:

- `*_cells.csv`: `cell_id`, level-0 `x`/`y`, and optional quality/feature references.
- `*_tiles.csv`: tile coordinates, brightness statistics, and filter decisions.
- `*_profile.json`, `*_timings.csv`, `*_summary.csv`: configuration and runtime profiling.
- `*_features.npy` plus metadata files when feature extraction is enabled.
- `failed_inputs.csv` for failures during a multi-input dataset run.

The current output does not contain a biological `cell_type` prediction.

## 7. Command-line equivalent

After `uv sync --locked`, the package also exposes the `wsi2sc` command:

```powershell
.\.venv\Scripts\wsi2sc.exe --help
.\.venv\Scripts\wsi2sc.exe _data\bonemarrowwsi_pediatricleukemia\A6BBC91AE73DD21C0533F735470A9CD0\MS -o outputs\wsi2sc
```

For reproducible experiments, prefer a reviewed YAML configuration over a long command line.

## 8. Handoff to cell-type classification

The natural extension point is after crop quality filtering or feature extraction:

1. Match DICOM annotation coordinates and labels to detected cell coordinates.
2. Train a supervised classifier on accepted crops or DinoBloom embeddings.
3. Add `cell_type`, class probabilities, and model metadata to the cell table.
4. Use retained level-0 coordinates to visualize predictions on the WSI.

Keep classification code in the main repository (or a separate package) rather than editing the pinned submodule unless the functionality belongs upstream in `wsi2sc`.